# Análise de Localização — Unitree Go2

**Disciplina:** Ciência de Dados — FEI Mestrado  
**Descrição:** Análise da qualidade de localização do RTABMAP utilizando uma ou duas câmeras no robô Go2.

---

### Fontes de Dados
| Arquivo | Descrição |
|---------|-----------|
| `localization_log.csv` | Métricas por atualização dos tópicos `/rtabmap/info` e `/localization_pose` (inliers, covariância, pose, etc.) |
| `plan_log.csv` | Poses do caminho planejado pelo tópico `/plan`, agrupadas por `plan_id` |

### Seções
1. **Carregamento e Processamento dos Dados** — leitura dos arquivos CSV e pré-processamento
2. **Visualização dos Caminhos** — caminho planejado vs trajetória real do robô
3. **Qualidade da Localização** — inliers, razão de hipótese e covariância ao longo do tempo

### 1. Carregamento e Processamento dos Dados

Os dados foram coletados em 60 execuções do robô Go2, divididas em dois grupos:
- **Runs 1–30:** robô equipado com **duas câmeras**
- **Runs 31–60:** robô equipado com **uma câmera**

Cada run possui dois arquivos de log: um com as métricas de localização e outro com o caminho planejado pelo stack de navegação NAV2.

In [31]:
import math
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import interp1d
from scipy.stats.mstats import winsorize
from plotly.subplots import make_subplots

DEBUG = False  # Set to True to print intermediate values


logs_dir = Path("localization_analysis/data/logs_cov_wrong")

loc_csv_paths = sorted(logs_dir.glob("logger_csv_*/localization_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_loc_df    = [pd.read_csv(loc_path) for loc_path in loc_csv_paths] 

plan_csv_paths = sorted(logs_dir.glob("logger_csv_*/plan_log.csv"), key=lambda p: int(p.parent.name.split('_')[-1]))
runs_plan_df    = [pd.read_csv(plan_path) for plan_path in plan_csv_paths] 

In [32]:
print(f' Localization Dataframes: {len(runs_loc_df)}\n Plan Dataframes {len(runs_plan_df)}') # Number of DF

 Localization Dataframes: 60
 Plan Dataframes 60


In [33]:
# Take the first path that NAV2 stack calculated (the plan_id = 1)
# Exception: run 5 uses plan_id = 13 (plan_id=1-12 starts at an outlier position — undocking)
plans_df = [df[df['plan_id'] == (13 if i == 4 else df['plan_id'].min())] for i, df in enumerate(runs_plan_df)]
plans_df[4]

,plan_id,pose_index,x,y
180,13,0,-2.8519,2.1549
181,13,1,-2.8019,2.1049
182,13,2,-2.7519,2.0549
183,13,3,-2.7019,2.0049
184,13,4,-2.6519,1.9549
...,...,...,...,...
328,13,148,1.7117,1.5549
329,13,149,1.7184,1.6049
330,13,150,1.7269,1.6549
331,13,151,1.7370,1.7049


In [34]:
# Drop rows where pose was not yet received (NaN)
paths_df = [df.dropna(subset=['pos_x', 'pos_y']) for df in runs_loc_df]
paths_df

[    timestamp_sec camera_mode  node_id  inliers  matches  inlier_ratio  \
 0    1.775683e+09      double    20082        0        0        0.0000   
 1    1.775683e+09      double    20083        0        0        0.0000   
 2    1.775683e+09      double    20084        0        0        0.0000   
 3    1.775683e+09      double    20085        0        0        0.0000   
 4    1.775683e+09      double    20086        0        0        0.0000   
 5    1.775683e+09      double    20087        0        0        0.0000   
 6    1.775683e+09      double    20088        0        0        0.0000   
 7    1.775683e+09      double    20089        0        0        0.0000   
 8    1.775683e+09      double    20090        0        0        0.0000   
 9    1.775683e+09      double    20091        3       43        0.0079   
 10   1.775683e+09      double    20092        3       29        0.0066   
 11   1.775683e+09      double    20093        3       28        0.0065   
 12   1.775683e+09      d

### 2. Visualização dos Caminhos

Comparação entre o caminho planejado pelo NAV2 e a trajetória real percorrida pelo robô em cada execução.

In [35]:
n_runs = len(paths_df)
cols = 2
rows = math.ceil(n_runs / cols)
titles = [f'Run {i+1}' for i in range(n_runs)]

fig = make_subplots(rows=rows, cols=cols,
                    subplot_titles=titles,
                    shared_yaxes=False)

for i, (actual_df, last_plan) in enumerate(zip(paths_df, plans_df)):
    row = i // cols + 1
    col = i % cols + 1

    fig.add_trace(go.Scatter(
        x=last_plan['x'], y=last_plan['y'],
        mode='lines', name='Planned path',
        line=dict(color='royalblue', dash='dash', width=2),
        legendgroup='plan', showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=actual_df['pos_x'], y=actual_df['pos_y'],
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=2),
        showlegend=(i == 0),
        legendgroup=f'run{i+1}'
    ), row=row, col=col)

fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(
    title='Planned vs Actual Path',
    hovermode='closest',
    height=500 * rows,
    width=900
)
fig.show()


#### 2.1 Normalização e Mediana dos Caminhos

Para comparar os caminhos entre diferentes runs, é necessário normalizá-los. Cada execução possui um número diferente de amostras e uma velocidade diferente, então não é possível comparar ponto a ponto diretamente.

A abordagem utilizada é a **parametrização por comprimento de arco**:
1. **Winsorize** — remove poses extremas (outliers do docking)
2. **Comprimento de arco normalizado** — mapeia cada ponto para `[0, 1]` proporcionalmente à distância percorrida
3. **Interpolação** — reamostrar todos os caminhos para `n_points` pontos uniformes no parâmetro de arco

Com isso, todos os caminhos ficam no mesmo espaço e é possível calcular a **mediana ponto a ponto** para os dfs de 1 e 2 câmeras.

##### 2.1.1 Caminhos Planejados para duas câmeras (Plans)

Aplicamos a normalização nos caminhos planejados pelo NAV2. Como os planos são gerados com amostras equidistantes, o efeito da interpolação é sutil, mas é necessário para manter o mesmo pipeline dos caminhos reais.

In [36]:
def arc_length_parametrize(pos_x, pos_y):
    coords = np.column_stack([pos_x, pos_y])
    deltas = np.diff(coords, axis=0) # diff in each point (x)
    arc = np.concatenate([[0], np.cumsum(np.hypot(deltas[:, 0], deltas[:, 1]))])

    last_element = arc[-1]
    arc_norm = arc / last_element

    if DEBUG:
        print(f"[arc_length_parametrize]")
        print(f"  n_points      : {len(pos_x)}")
        print(f"  total length  : {last_element:.4f} m")
        print(f"  arc[:5]  : {arc[:5]}")
        print(f"  arc[-5:] : {arc[-5:]}")
        print(f"  arc_norm[:5]  : {arc_norm[:5]}")
        print(f"  arc_norm[-5:] : {arc_norm[-5:]}")

    return arc_norm

def plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Interpolation check'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(u, pos_x, 'o', t, fx(t), '-')
    ax1.set_title('X vs arc_norm')
    ax1.set_xlabel('arc_norm (0→1)')
    ax2.plot(u, pos_y, 'o', t, fy(t), '-')
    ax2.set_title('Y vs arc_norm')
    ax2.set_xlabel('arc_norm (0→1)')
    plt.suptitle(f'[DEBUG] {title}')
    plt.tight_layout()
    plt.show()

def winsorize_and_resample(df, x_col='pos_x', y_col='pos_y', limits=(0.013, 0.013), n_points=500, idx=None):
    if idx is not None and idx >= 30:
        if DEBUG:
            print(f"[winsorize_and_resample — plans] skipped run {idx} (single camera — will process later)")
        return None

    pos_x = np.array(winsorize(df[x_col], limits=limits))
    pos_y = np.array(winsorize(df[y_col], limits=limits))

    if DEBUG:
        print(f"[winsorize_and_resample — plans]")
        print(f"  input rows    : {len(df)}")
        print(f"  limits        : {limits}")
        print(f"  pos_x range   : [{pos_x.min():.4f}, {pos_x.max():.4f}]")
        print(f"  pos_y range   : [{pos_y.min():.4f}, {pos_y.max():.4f}]")

    u = arc_length_parametrize(pos_x, pos_y)

    t = np.linspace(0, 1, n_points)
    fx = interp1d(u, pos_x, kind='linear')
    fy = interp1d(u, pos_y, kind='linear')

    # If you want to plot the graph (X,Arc_Param)
    if DEBUG:
        print(f"  resampled to  : {n_points} points")
        plot_interpolation_check(u, pos_x, pos_y, t, fx, fy, title='Plans')

    return fx(t), fy(t)

# Plan paths (use x/y columns)
plans_resampled_2cam = [winsorize_and_resample(df, x_col='x', y_col='y', idx=i) for i, df in enumerate(plans_df)]

plans_xs_2cam = np.array([p[0] for p in plans_resampled_2cam if p is not None])
plans_ys_2cam = np.array([p[1] for p in plans_resampled_2cam if p is not None])
median_plan_x_2cam = np.median(plans_xs_2cam, axis=0)
median_plan_y_2cam = np.median(plans_ys_2cam, axis=0)

**Verificação — Planos 2 câmeras:** comparação entre dados brutos e após reamostagem por comprimento de arco.

In [37]:
_df = plans_df[0]
_raw_x = _df['x'].values
_raw_y = _df['y'].values

_wins_x = np.array(winsorize(_df['x'], limits=(0.013, 0.013)))
_wins_y = np.array(winsorize(_df['y'], limits=(0.013, 0.013)))

_interp_x, _interp_y = plans_resampled_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Plano 1 — Etapas de Processamento', hovermode='closest', width=1300)
fig.show()

**Planos 2 câmeras — todos os planos reamostrados** com mediana calculada ponto a ponto.

In [38]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Median plan',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Plans + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

##### 2.1.2 Trajetórias Reais para duas câmeras (Paths)

Aplicamos o mesmo pipeline nos caminhos reais percorridos pelo robô. Aqui o winsorize é mais importante: o robô pode ter poses ruins no início ou fim da execução (ex: antes de sair do docking).

In [39]:
# Trajetórias reais — duas câmeras (winsorize + interpolação)
def _process_path_winsorize(df, n_points=500):
    pos_x = np.array(winsorize(df['pos_x'], limits=(0.28, 0.00)))
    pos_y = np.array(winsorize(df['pos_y'], limits=(0.0, 0.32)))
    u = arc_length_parametrize(pos_x, pos_y)
    t = np.linspace(0, 1, n_points)
    return interp1d(u, pos_x)(t), interp1d(u, pos_y)(t)

paths_winsorized_2cam = [_process_path_winsorize(df) for df in paths_df[:30]]
paths_xs_winsorized_2cam = np.array([p[0] for p in paths_winsorized_2cam])
paths_ys_winsorized_2cam = np.array([p[1] for p in paths_winsorized_2cam])
median_path_winsorized_x_2cam = np.median(paths_xs_winsorized_2cam, axis=0)
median_path_winsorized_y_2cam = np.median(paths_ys_winsorized_2cam, axis=0)


**Verificação — Trajetórias 2 câmeras (winsorize):** comparação entre sem processamento, winsorize e após interpolação.

In [40]:
_df = paths_df[0]
_raw_x = _df['pos_x'].values
_raw_y = _df['pos_y'].values

_wins_x = np.array(winsorize(_df['pos_x'], limits=(0.30, 0.00)))
_wins_y = np.array(winsorize(_df['pos_y'], limits=(0.0, 0.32)))

_interp_x, _interp_y = paths_winsorized_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Winsorize",
    "Winsorize + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_wins_x, y=_wins_y,
    mode='markers', name='Winsorize',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_interp_x, y=_interp_y,
    mode='markers', name='Winsorize + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Trajetória 1 — Etapas de Processamento', hovermode='closest', width=1300)
fig.show()

**Trajetórias 2 câmeras (winsorize) — todas as runs** com mediana calculada.

In [41]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in paths_winsorized_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name=f'Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_winsorized_x_2cam, y=median_path_winsorized_y_2cam,
    mode='lines', name='Median path',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Paths + Median',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

##### 2.1.3 Alternativa ao Winsorize — Clip pela Geometria do Plano

O winsorize remove outliers por **percentil de valor de coordenada**, de forma independente em X e Y. Isso funciona bem quando os outliers estão distribuídos simetricamente nos extremos das coordenadas.

No entanto, neste experimento alguns robôs **começaram a se mover antes do início do plano calculado** (ex: saindo do docking). Esses pontos extras estão numa direção diagonal em relação ao plano e não são removidos corretamente pelo winsorize, pois ele não considera a geometria do percurso.

**Solução:** para cada trajetória, encontrar o ponto mais próximo do **início** e do **fim da mediana do plano**, e descartar tudo fora desse intervalo. Isso garante que apenas o trecho em que o robô estava executando o plano é considerado, independente da direção do movimento.

In [42]:
def clip_to_plan(pos_x, pos_y, plan_x, plan_y, max_dist=0.12, run_idx=None):
    dists_start = np.hypot(pos_x - plan_x[0], pos_y - plan_y[0])
    dists_end   = np.hypot(pos_x - plan_x[-1], pos_y - plan_y[-1])
    idx_start = np.argmin(dists_start)
    idx_end   = np.argmin(dists_end)

    label = f"Run {run_idx + 1}" if run_idx is not None else "Run ?"
    if dists_start[idx_start] > max_dist:
        print(f"[WARN] {label} — início distante do plano: {dists_start[idx_start]:.2f}m")
    if dists_end[idx_end] > max_dist:
        print(f"[WARN] {label} — fim distante do plano: {dists_end[idx_end]:.2f}m")

    return pos_x[idx_start:idx_end + 1], pos_y[idx_start:idx_end + 1]


def resample_clipped(df, plan_x, plan_y, n_points=500, run_idx=None):
    pos_x = df['pos_x'].values.astype(float)
    pos_y = df['pos_y'].values.astype(float)

    pos_x, pos_y = clip_to_plan(pos_x, pos_y, plan_x, plan_y, run_idx=run_idx)

    if len(pos_x) < 2:
        return None

    u = arc_length_parametrize(pos_x, pos_y)
    t = np.linspace(0, 1, n_points)
    fx = interp1d(u, pos_x, kind='linear')
    fy = interp1d(u, pos_y, kind='linear')
    return fx(t), fy(t)


paths_clipped_2cam = [resample_clipped(df, median_plan_x_2cam, median_plan_y_2cam, run_idx=i) for i, df in enumerate(paths_df[:30])]

paths_xs_clipped_2cam = np.array([p[0] for p in paths_clipped_2cam if p is not None])
paths_ys_clipped_2cam = np.array([p[1] for p in paths_clipped_2cam if p is not None])
median_path_x_2cam = np.median(paths_xs_clipped_2cam, axis=0)
median_path_y_2cam = np.median(paths_ys_clipped_2cam, axis=0)

[WARN] Run 1 — início distante do plano: 0.18m
[WARN] Run 3 — início distante do plano: 0.15m
[WARN] Run 4 — início distante do plano: 0.13m
[WARN] Run 11 — início distante do plano: 0.16m
[WARN] Run 26 — início distante do plano: 0.13m
[WARN] Run 27 — início distante do plano: 0.17m
[WARN] Run 29 — início distante do plano: 0.15m


**Verificação — Trajetórias 2 câmeras (clip):** comparação entre sem processamento, clip pelo plano e após interpolação.

In [43]:
_df = paths_df[0]
_raw_x = _df['pos_x'].values
_raw_y = _df['pos_y'].values

_clip_only_x, _clip_only_y = clip_to_plan(_raw_x, _raw_y, median_plan_x_2cam, median_plan_y_2cam)
_clip_x, _clip_y           = paths_clipped_2cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Clip pelo Plano",
    "Clip + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_clip_only_x, y=_clip_only_y,
    mode='markers', name='Clip pelo Plano',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_clip_x, y=_clip_y,
    mode='markers', name='Clip + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Trajetória 1 — Etapas do Clip pelo Plano', hovermode='closest', width=1300)
fig.show()

[WARN] Run ? — início distante do plano: 0.18m


**Trajetórias 2 câmeras (clip) — todas as runs** com mediana calculada.

In [44]:
fig = go.Figure()

for i, (rx, ry) in enumerate(p for p in paths_clipped_2cam if p is not None):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name='Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_x_2cam, y=median_path_y_2cam,
    mode='lines', name='Mediana das trajetórias',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Trajetórias (Clip pelo Plano) + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

**Duas câmeras — mediana final:** sobreposição da mediana do plano calculado com a mediana da trajetória real.

In [45]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_x_2cam, y=median_path_y_2cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Duas Câmeras — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
fig.show()

##### MSE — Duas Câmeras

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados**. Um único número que resume o quanto o robô desviou do plano em média ao longo do percurso.

In [46]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_2cam, median_path_x_2cam)
mse_y = mean_squared_error(median_plan_y_2cam, median_path_y_2cam)
mse_2cam  = (mse_x + mse_y) / 2
rmse_2cam = np.sqrt(mse_2cam)
print(f'MSE  (mediana trajetória vs mediana plano) — duas câmeras: {mse_2cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — duas câmeras: {rmse_2cam:.4f} m')


MSE  (mediana trajetória vs mediana plano) — duas câmeras: 0.013307 m²
RMSE (mediana trajetória vs mediana plano) — duas câmeras: 0.1154 m


#### 2.2 Uma Câmera (Runs 31–60)

Repetimos o mesmo pipeline para as 30 execuções com uma câmera. As funções são idênticas — apenas o conjunto de dados muda.

##### 2.2.1 Caminhos Planejados

In [47]:
plans_resampled_1cam = [winsorize_and_resample(df, x_col='x', y_col='y') for df in plans_df[30:]]

plans_xs_1cam = np.array([p[0] for p in plans_resampled_1cam if p is not None])
plans_ys_1cam = np.array([p[1] for p in plans_resampled_1cam if p is not None])
median_plan_x_1cam = np.median(plans_xs_1cam, axis=0)
median_plan_y_1cam = np.median(plans_ys_1cam, axis=0)

In [48]:
fig = go.Figure()

for i, (px, py) in enumerate(p for p in plans_resampled_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=px, y=py,
        mode='lines', name='Plans',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='plans', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Mediana dos planos',
    line=dict(color='royalblue', dash='dash', width=3)
))

fig.update_layout(
    title='Planos — Uma Câmera + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

##### 2.2.2 Trajetórias Reais (Clip pelo Plano)

In [49]:
paths_clipped_1cam = [
    resample_clipped(df, median_plan_x_1cam, median_plan_y_1cam, run_idx=i + 30)
    for i, df in enumerate(paths_df[30:])
]

paths_xs_clipped_1cam = np.array([p[0] for p in paths_clipped_1cam if p is not None])
paths_ys_clipped_1cam = np.array([p[1] for p in paths_clipped_1cam if p is not None])
median_path_x_1cam = np.median(paths_xs_clipped_1cam, axis=0)
median_path_y_1cam = np.median(paths_ys_clipped_1cam, axis=0)

[WARN] Run 33 — início distante do plano: 0.14m
[WARN] Run 36 — início distante do plano: 0.13m
[WARN] Run 37 — início distante do plano: 0.13m
[WARN] Run 39 — início distante do plano: 0.13m
[WARN] Run 53 — início distante do plano: 0.15m
[WARN] Run 58 — início distante do plano: 0.16m
[WARN] Run 59 — início distante do plano: 0.16m


Verificação visual da interpolação: o gráfico abaixo mostra o caminho feito 1 antes e depois da reamostragem por comprimento de arco + clip + interpolação linear. Os pontos originais (vermelho) devem ser cobertos pela curva interpolada (azul).

In [50]:
_df = paths_df[30]
_raw_x = _df['pos_x'].values
_raw_y = _df['pos_y'].values

_clip_only_x, _clip_only_y = clip_to_plan(_raw_x, _raw_y, median_plan_x_1cam, median_plan_y_1cam)
_clip_x, _clip_y           = paths_clipped_1cam[0]

fig = make_subplots(rows=1, cols=3, subplot_titles=[
    "Sem processamento",
    "Clip pelo Plano",
    "Clip + Interpolação"
])

fig.add_trace(go.Scatter(
    x=_raw_x, y=_raw_y,
    mode='markers', name='Raw',
    marker=dict(color='gray', size=4)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=_clip_only_x, y=_clip_only_y,
    mode='markers', name='Clip pelo Plano',
    marker=dict(color='tomato', size=4)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=_clip_x, y=_clip_y,
    mode='markers', name='Clip + Interpolação',
    marker=dict(color='royalblue', size=4)
), row=1, col=3)

fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_yaxes(scaleanchor='x3', row=1, col=3)
fig.update_layout(title='Trajetória 30 — Etapas do Clip pelo Plano', hovermode='closest', width=1300)
fig.show()

Verificação visual da interpolação: o gráfico abaixo mostra todos os caminhos e a mediana calculada a partir das etapas de preprocessamentos citadas anteriormente.

In [51]:
fig = go.Figure()

for i, (rx, ry) in enumerate(p for p in paths_clipped_1cam if p is not None):
    fig.add_trace(go.Scatter(
        x=rx, y=ry,
        mode='lines', name='Runs',
        line=dict(color='tomato', width=1),
        opacity=0.3,
        legendgroup='runs', showlegend=(i == 0)
    ))

fig.add_trace(go.Scatter(
    x=median_path_x_1cam, y=median_path_y_1cam,
    mode='lines', name='Mediana das trajetórias',
    line=dict(color='black', width=2.5)
))

fig.update_layout(
    title='Trajetórias — Uma Câmera (Clip pelo Plano) + Mediana',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

Por fim será demonstrado gráficamente a junção das medianas tanto do caminho calculado, quanto o caminho feito.

In [52]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode="lines", name="Mediana do plano calculado",
    line=dict(color="royalblue", dash="dash", width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_x_1cam, y=median_path_y_1cam,
    mode="lines", name="Mediana do caminho feito",
    line=dict(color="tomato", width=2.5)
))

fig.update_layout(
    title="Uma Câmera — Mediana do Plano vs Mediana da Trajetória",
    xaxis_title="X (m)", yaxis_title="Y (m)",
    yaxis_scaleanchor="x", hovermode="closest"
)
fig.show()

##### MSE — Uma Câmera

Desvio entre a **mediana das trajetórias reais** e a **mediana dos planos calculados** para as runs com uma câmera.

In [53]:
from sklearn.metrics import mean_squared_error

mse_x = mean_squared_error(median_plan_x_1cam, median_path_x_1cam)
mse_y = mean_squared_error(median_plan_y_1cam, median_path_y_1cam)
mse_1cam  = (mse_x + mse_y) / 2
rmse_1cam = np.sqrt(mse_1cam)
print(f'MSE  (mediana trajetória vs mediana plano) — uma câmera: {mse_1cam:.6f} m²')
print(f'RMSE (mediana trajetória vs mediana plano) — uma câmera: {rmse_1cam:.4f} m')

MSE  (mediana trajetória vs mediana plano) — uma câmera: 0.011655 m²
RMSE (mediana trajetória vs mediana plano) — uma câmera: 0.1080 m


#### 2.3 Comparação entre Grupos

Análise comparativa entre os dois grupos de execução — robô com duas câmeras (runs 1–30) e com uma câmera (runs 31–60) — dividida em duas etapas: análise gráfica das trajetórias e análise quantitativa via MSE/RMSE.

##### 2.3.1 Análise Gráfica

Comparação visual das trajetórias medianas dos dois grupos em relação ao plano mediano global.

In [54]:
fig = make_subplots(rows=1, cols=2, subplot_titles=['Duas Câmeras', 'Uma Câmera'])

# Duas câmeras
fig.add_trace(go.Scatter(
    x=median_plan_x_2cam, y=median_plan_y_2cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=median_path_x_2cam, y=median_path_y_2cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='royalblue', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=1)

# Uma câmera
fig.add_trace(go.Scatter(
    x=median_plan_x_1cam, y=median_plan_y_1cam,
    mode='lines', name='Plano mediano',
    line=dict(color='gray', dash='dash', width=2),
    legendgroup='plan', showlegend=True
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=median_path_x_1cam, y=median_path_y_1cam,
    mode='lines', name='Mediana trajetória',
    line=dict(color='tomato', width=2.5),
    legendgroup='path', showlegend=True
), row=1, col=2)

fig.update_yaxes(scaleanchor='x',  row=1, col=1)
fig.update_yaxes(scaleanchor='x2', row=1, col=2)
fig.update_xaxes(title_text='X (m)')
fig.update_yaxes(title_text='Y (m)', row=1, col=1)
fig.update_layout(
    title='Comparação: Mediana do Plano vs Mediana da Trajetória',
    hovermode='closest', width=1100
)
fig.show()

Para a comparação final, calcula-se a **mediana global do caminho planejado** — combinando os planos de ambos os grupos (duas e uma câmera). Com isso é possível visualizar num único gráfico o quanto cada grupo desviou em relação ao mesmo plano de referência.

In [55]:
# Mediana global do plano (une os dois grupos)
all_plans_xs = np.vstack([plans_xs_2cam, plans_xs_1cam])
all_plans_ys = np.vstack([plans_ys_2cam, plans_ys_1cam])
median_plan_x_global = np.median(all_plans_xs, axis=0)
median_plan_y_global = np.median(all_plans_ys, axis=0)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=median_plan_x_global, y=median_plan_y_global,
    mode='lines', name='Plano mediano (global)',
    line=dict(color='gray', dash='dash', width=2)
))

fig.add_trace(go.Scatter(
    x=median_path_x_2cam, y=median_path_y_2cam,
    mode='lines', name='Mediana trajetória — duas câmeras',
    line=dict(color='royalblue', width=2.5)
))

fig.add_trace(go.Scatter(
    x=median_path_x_1cam, y=median_path_y_1cam,
    mode='lines', name='Mediana trajetória — uma câmera',
    line=dict(color='tomato', width=2.5)
))

fig.update_layout(
    title='Plano Mediano Global vs Medianas das Trajetórias',
    xaxis_title='X (m)', yaxis_title='Y (m)',
    yaxis_scaleanchor='x', hovermode='closest'
)
fig.show()

Como podemos ver, os dois caminhos ficaram bem parecidos, o que acabei ficando um pouco surpreso. Uma análise que pode nos dar uma informação mais precisa é o MSE e RMSE que calculamos anteriormente para os dois casos, esses valores quantificam exatamente o desvio médio de cada grupo em relação ao plano, independentemente da similaridade visual.

In [56]:
print('=' * 45)
print(f'  MSE  — duas câmeras : {mse_2cam:.6f} m²')
print(f'  RMSE — duas câmeras : {rmse_2cam:.4f} m')
print('-' * 45)
print(f'  MSE  — uma câmera   : {mse_1cam:.6f} m²')
print(f'  RMSE — uma câmera   : {rmse_1cam:.4f} m')
print('=' * 45)
diff_rmse = abs(rmse_2cam - rmse_1cam)
better = 'duas câmeras' if rmse_2cam < rmse_1cam else 'uma câmera'
print(f'  Diferença RMSE      : {diff_rmse:.4f} m')
print(f'  Grupo mais preciso  : {better}')
print('=' * 45)

  MSE  — duas câmeras : 0.013307 m²
  RMSE — duas câmeras : 0.1154 m
---------------------------------------------
  MSE  — uma câmera   : 0.011655 m²
  RMSE — uma câmera   : 0.1080 m
  Diferença RMSE      : 0.0074 m
  Grupo mais preciso  : uma câmera


##### 2.3.2 Análise Quantitativa — MSE e RMSE

O MSE e RMSE medem numericamente o desvio médio de cada grupo em relação ao plano. O boxplot complementa mostrando a dispersão dos erros entre as runs individuais — revelando se o desvio é consistente ou se há execuções muito discrepantes.

In [57]:
# RMSE de cada run individualmente vs plano mediano global
rmse_runs_2cam = []
for rx, ry in (p for p in paths_clipped_2cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_2cam.append(np.sqrt(np.mean(d)))

rmse_runs_1cam = []
for rx, ry in (p for p in paths_clipped_1cam if p is not None):
    d = (rx - median_plan_x_global)**2 + (ry - median_plan_y_global)**2
    rmse_runs_1cam.append(np.sqrt(np.mean(d)))

fig = go.Figure()
fig.add_trace(go.Box(
    y=rmse_runs_2cam, name='Duas câmeras',
    marker_color='royalblue', boxpoints='all', jitter=0.3, pointpos=-1.5
))
fig.add_trace(go.Box(
    y=rmse_runs_1cam, name='Uma câmera',
    marker_color='tomato', boxpoints='all', jitter=0.3, pointpos=-1.5
))
fig.update_layout(
    title='Dispersão do RMSE por Run — Duas Câmeras vs Uma Câmera',
    yaxis_title='RMSE (m)', hovermode='closest'
)
fig.show()

### 3. Qualidade da Localização

Análise das métricas internas do RTABMAP para cada grupo. Os dados são agrupados por câmera e comparados via boxplot (quando possível) para revelar diferenças na qualidade de localização visual, confiança da estimativa e custo computacional. As métricas de localização disponíveis são:

**Qualidade do casamento visual (RANSAC / odometria visual):**
- **`inliers`** — número de correspondências de features consideradas geometricamente consistentes pelo RANSAC ao estimar a transformação entre frames. Valores altos indicam que a cena tem textura suficiente e que o casamento visual foi bem-sucedido.
- **`matches`** — total de correspondências de features encontradas antes da filtragem geométrica. Reflete a riqueza visual da cena.
- **`inlier_ratio`** = `inliers / matches` — fração de correspondências válidas. É o melhor indicador *normalizado* da qualidade do casamento: valores baixos sugerem cenas pouco texturizadas, oclusões ou movimento brusco.

**Confiança do reconhecimento de lugar (loop closure):**
- **`hypothesis_ratio`** — razão entre a melhor hipótese de loop closure e o limiar de aceitação. Valores == 1 implicam que o RTABMAP fechou o laço, possuindo grande confiança do lugar; valores baixos indicam que o reconhecimento de lugar não está confiante.
- **`loop_closure_id`** — id do nó com o qual ocorreu o fechamento de laço (`-1` quando não houve). Útil para contar eventos de relocalização ao longo da trajetória.

**Incerteza da pose estimada:**
- **`cov_xx`, `cov_yy`, `cov_yaw`** — variâncias diagonais da matriz de covariância da pose. Quanto menores, mais confiante está o filtro sobre a estimativa.
- **`cov_pos_trace`** = `cov_xx + cov_yy` — traço da submatriz de posição; resume em um único escalar a incerteza translacional da estimativa.

**Custo computacional e gerenciamento de memória:**
- **`detection_time_ms`** — tempo gasto pelo módulo de detecção de loop closure por atualização.
- **`total_time_ms`** — tempo total de processamento do RTABMAP por atualização (detecção + manutenção do grafo + memória).
- **`wm_size`** — tamanho da *Working Memory* (número de nós ativos do grafo). Cresce com a exploração e impacta diretamente o custo computacional.

#### 3.1 Preparação dos Dados

Agregamos todos os logs de localização separados por grupo para facilitar a comparação.

In [58]:
# Concatena todos os runs de cada grupo com label
loc_2cam_raw = pd.concat(
    [df.assign(run=i+1) for i, df in enumerate(runs_loc_df[:30])],
    ignore_index=True
)
loc_1cam_raw = pd.concat(
    [df.assign(run=i+31) for i, df in enumerate(runs_loc_df[30:])],
    ignore_index=True
)

Plot do histograma para ver a distribuição dos dados e ter uma ideia de como estão, se possuem outliers.

In [59]:
metrics = ['inlier_ratio', 'hypothesis_ratio', 'cov_pos_trace', 'detection_time_ms']

fig = make_subplots(
    rows=len(metrics), cols=2,
    column_titles=['Duas Câmeras', 'Uma Câmera'],
    row_titles=metrics,
    vertical_spacing=0.06
)

for row, metric in enumerate(metrics, start=1):
    fig.add_trace(go.Histogram(
        x=loc_2cam_raw[metric], name=metric,
        marker_color='royalblue', opacity=0.7,
        showlegend=False
    ), row=row, col=1)
    fig.add_trace(go.Histogram(
        x=loc_1cam_raw[metric], name=metric,
        marker_color='tomato', opacity=0.7,
        showlegend=False
    ), row=row, col=2)

fig.update_layout(
    title='Distribuição das Métricas — Duas Câmeras vs Uma Câmera',
    height=300 * len(metrics), width=900,
    hovermode='closest'
)
fig.show()

Observando os histogramas, é nítida a presença de outliers em duas métricas: `inlier_ratio` e `cov_pos_trace`. A seguir é feita uma análise para tentar explicar esses outliers e avaliar se é possível removê-los ou se representam alguma informação relevante sobre a localização durante a trajetória do robô.

**Hipótese 1 — `inlier_ratio = 0`:** correspondem a momentos em que há perda de frames das câmeras ou ao início da run, com o robô ainda parado no docking, em que o RTABMAP ainda não processa correspondências visuais.

**Hipótese 2 — `cov_pos_trace > 100`:** para validar essa hipótese é necessário analisar a covariância ao longo do tempo, identificando em quais instantes da trajetória esses valores extremos aparecem (provavelmente concentrados antes do primeiro loop closure).

Para verificar ambas as hipóteses, plotamos abaixo `inlier_ratio` e `cov_pos_trace` em função do tempo relativo ao início de cada run, separando por modo de câmera.

In [60]:
# Análise temporal dos outliers — usa os dados brutos (sem filtro) para
# revelar EM QUE INSTANTE da trajetória os valores extremos aparecem.

import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _concat_with_rel_time(runs_subset, run_offset):
    parts = []
    for i, df in enumerate(runs_subset):
        d = df.copy()
        d['run']   = i + run_offset
        d['t_rel'] = d['timestamp_sec'] - d['timestamp_sec'].min()
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

_loc_2cam_t = _concat_with_rel_time(runs_loc_df[:30], 1)
_loc_1cam_t = _concat_with_rel_time(runs_loc_df[30:], 31)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'inlier_ratio — Duas Câmeras', 'inlier_ratio — Uma Câmera',
        'cov_pos_trace — Duas Câmeras', 'cov_pos_trace — Uma Câmera',
    ),
    vertical_spacing=0.13, horizontal_spacing=0.08,
)

for df_g, color, col in [(_loc_2cam_t, 'royalblue', 1),
                         (_loc_1cam_t, 'tomato',    2)]:
    for run_id, df_run in df_g.groupby('run'):
        fig.add_trace(go.Scatter(
            x=df_run['t_rel'], y=df_run['inlier_ratio'],
            mode='markers', marker=dict(color=color, size=3),
            opacity=0.45, showlegend=False,
            hovertemplate=f'run {run_id}<br>t=%{{x:.1f}}s<br>inlier_ratio=%{{y:.2f}}<extra></extra>'
        ), row=1, col=col)
        fig.add_trace(go.Scatter(
            x=df_run['t_rel'], y=df_run['cov_pos_trace'],
            mode='markers', marker=dict(color=color, size=3),
            opacity=0.45, showlegend=False,
            hovertemplate=f'run {run_id}<br>t=%{{x:.1f}}s<br>cov_pos_trace=%{{y:.3f}}<extra></extra>'
        ), row=2, col=col)

# Limiar usado pelo filtro: cov_pos_trace = 100
for col in (1, 2):
    fig.add_hline(
        y=100, line_dash='dash', line_color='gray',
        row=2, col=col,
        annotation_text='COV_MAX = 100',
        annotation_position='top right',
    )

# Eixo Y em escala log para cov_pos_trace — os valores vão de ~1e-3 até o sentinel 19998
fig.update_yaxes(type='log', row=2, col=1)
fig.update_yaxes(type='log', row=2, col=2)

for col in (1, 2):
    fig.update_xaxes(title_text='Tempo relativo ao início da run (s)', row=2, col=col)
fig.update_yaxes(title_text='inlier_ratio',        row=1, col=1)
fig.update_yaxes(title_text='cov_pos_trace (log)', row=2, col=1)

fig.update_layout(
    title='Outliers ao longo da trajetória — inlier_ratio e cov_pos_trace vs. tempo',
    height=720, width=1100, hovermode='closest',
)
fig.show()

#### Observação — origem dos outliers e impacto na comparação 1 × 2 câmeras

Após inspecionar o gráfico acima, fica claro que os outliers (`inlier_ratio = 0` e `cov_pos_trace` muito alto) se concentram nos **primeiros segundos de cada run**. Investigando os tópicos do Go2 em tempo real, foi possível identificar a causa:

- Quando o robô sai do docking, sua pose é **setada externamente** pelo sistema de navegação.
- O RTABMAP, ao receber essa pose imposta, aparentemente **reinicia o módulo de localização**: a covariância volta para o valor de sentinel e a contagem de `inliers` cai a zero, como se o algoritmo precisasse reconstruir o vínculo visual com o mapa do zero.
- Conforme o robô se movimenta pelo ambiente, a covariância decai e o número de `inliers` sobe, comportamento consistente com o RTABMAP "reaprendendo" a posição a partir das observações visuais.

**Implicação para esta análise:** o ruído introduzido por esse comportamento contamina a comparação entre 1 e 2 câmeras, pois grande parte das amostras "ruins" não reflete a qualidade real do sistema de visão e sim o transitório de re-inicialização. Por isso, **esta análise (sobre os dados em `data/logs_cov_wrong/`) não é considerada conclusiva** para o objetivo do benchmark.
 
**Próximo passo:** uma nova coleta foi realizada com protocolo ajustado (dados em `logs/`), e toda a análise será refeita sobre esses dados na sequência do notebook, buscando uma comparação mais fiel entre os dois modos.